In [3]:
import sys
!{sys.executable} -m pip install pandas numpy mlxtend

In [4]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

items = ['rice', 'ghee', 'dal', 'sugar', 'oil', 'atta', 'salt', 'tea', 'soap', 'biscuit']

rows = []
for i in range(300):
    basket = []
    if random.random() < 0.75:
        basket.append('rice')
        if random.random() < 0.70:
            basket.append('ghee')
    if random.random() < 0.60:
        basket.append('dal')
        if random.random() < 0.65:
            basket.append('salt')
    extras = random.sample([x for x in items if x not in basket], k=random.randint(1, 3))
    basket.extend(extras)
    rows.append({'transaction_id': i+1, 'items': ', '.join(set(basket))})

df = pd.DataFrame(rows)
df.to_csv('sales_data.csv', index=False)
print("✅ CSV created successfully!")
print(df.head(10))

✅ CSV created successfully!
   transaction_id                                        items
0               1  sugar, dal, ghee, biscuit, salt, soap, rice
1               2      sugar, dal, oil, ghee, salt, soap, rice
2               3               dal, ghee, biscuit, salt, rice
3               4                       salt, ghee, sugar, dal
4               5                               salt, dal, tea
5               6                                          tea
6               7           dal, tea, oil, biscuit, salt, rice
7               8                      ghee, sugar, rice, soap
8               9             dal, oil, atta, ghee, salt, rice
9              10                 sugar, tea, ghee, salt, rice


In [5]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Reading the CSV file into a table
df = pd.read_csv('sales_data.csv')

# Splitting items like "rice, ghee, dal" into a list ['rice','ghee','dal']
transactions = [row.split(', ') for row in df['items']]

# Converting items into True/False columns (did customer buy this item?)
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets = apriori(df_encoded, min_support=0.15, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
rules = rules.sort_values('confidence', ascending=False)

print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

           antecedents consequents   support  confidence      lift
66         (ghee, oil)      (rice)  0.170000    1.000000  1.224490
72       (ghee, sugar)      (rice)  0.190000    0.982759  1.203378
85  (ghee, sugar, dal)      (rice)  0.150000    0.978261  1.197870
31     (ghee, biscuit)      (rice)  0.160000    0.960000  1.175510
75         (ghee, tea)      (rice)  0.203333    0.938462  1.149137
92    (ghee, dal, tea)      (rice)  0.170000    0.927273  1.135436
17              (ghee)      (rice)  0.603333    0.923469  1.130779
64         (salt, tea)       (dal)  0.156667    0.921569  1.256684
62        (salt, soap)       (dal)  0.150000    0.918367  1.252319
35         (ghee, dal)      (rice)  0.446667    0.911565  1.116202
69        (salt, ghee)      (rice)  0.340000    0.902655  1.105292
79   (salt, ghee, dal)      (rice)  0.293333    0.897959  1.099542
2               (atta)      (rice)  0.210000    0.887324  1.086519
5            (biscuit)      (rice)  0.220000    0.880000  1.07